In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy.stats import spearmanr, rankdata
from joblib import Parallel, delayed
import warnings
import gc
import argparse
from tqdm.auto import tqdm
import glob
import matplotlib.pyplot as plt
import networkx as nx
import re
import sys

In [2]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score

In [ ]:
twinfer_kwargs = {
    "path_to_simulation_file": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/fixed_z_simulations/Extrema_mutual_regulation/df_rows_0_0_0_16022026_060219_ncells_6000_Extrema_mutual_regulation_3_0_152fbcda.csv",
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 12, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": 'all-potential-regulation', #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 18,
    "match_sim_details": False
    }

correlation_matrices = infer_with_twinfer(
            **twinfer_kwargs
            )

In [5]:
#Path to TwINFER code repository
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_save_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/fixed_z/"
os.makedirs(path_to_save_plot_data, exist_ok = True)

# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    split_and_merge_simulations,
    get_param_data, 
    plot_matrix_as_heatmap,
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

In [ ]:
# ============================================================
# Fonts / plotting defaults
# ============================================================
import matplotlib.font_manager as fm

font_paths = [
    f"{path_to_code_repo}/fonts//Arial.ttf",
    f"{path_to_code_repo}/fonts//Arial Bold.ttf",
    f"{path_to_code_repo}/fonts//Arial Italic.ttf",
    f"{path_to_code_repo}/fonts//Arial Bold Italic.ttf",
]
for fp in font_paths:
    try:
        fm.fontManager.addfont(fp)
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 12.5
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.dpi'] = 400
plt.rcParams['axes.grid'] = False

In [7]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_five_gene = {
    'n_cells': 6000,
    'simulation_time_before_division': 2000,
    'twin_simulation_time_after_division': 48,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_fan_out.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,0,0]],
    "log_file": None,  # Will be updated per network type
    "type": "fan-out",  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 12
}

In [ ]:
def process_matrix(twinfer_kwargs, id):
    # Process each network type
    all_correlation_matrices = {}
    
    # Use the first CSV file found (or you can add logic to select specific one)        
    # Update config for this network type
    config = twinfer_kwargs['base_config']
    network_type = "fan-out" 

    def make_json_safe(obj):
        if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
            return obj.to_dict()
        if isinstance(obj, set):
            return list(obj)
        return obj
        
    # Check if required files exist
    if not os.path.exists(config["path_to_connectivity_matrix"]):
        print(f"Warning: Connectivity matrix not found for {network_type}")

    if not os.path.exists(config["param_csv"]):
        print(f"Warning: Parameter CSV not found for {network_type}")
    correlation_matrices = []
    try:
        print("running")
        # Run inference for this network type
        correlation_matrices = infer_with_twinfer(
            **twinfer_kwargs
            )
            

    except Exception as e:
        raise(f"Error processing {network_type}: {str(e)}")
    
    import json
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_save_plot_data}fan_out_{id}.json"
    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)
    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_save_plot_data}/filtered_directional_correlation_type_{network_type}_{id}.csv"
    gene_correlation_file_name = f"{path_to_save_plot_data}/gene_correlation_type_{network_type}_{id}.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
    return

path_to_simulations = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/fixed_z_simulations/Average_fan_out/"
prefix = "df_rows_0_0_0_"

list_simulations = [
    f.path
    for f in os.scandir(path_to_simulations)
    if f.is_file() and f.name.startswith(prefix)
]
len(list_simulations)

In [ ]:
for id, file_name in enumerate(list_simulations[:5]):
    twinfer_kwargs = {
    "path_to_simulation_file": file_name,
    "base_config": base_config_five_gene,
    "t1": 10,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 12, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": 'all-potential-regulation', #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 18,
    "match_sim_details": False
    }
    process_matrix(twinfer_kwargs, id)

In [ ]:
import json
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PATHS
# ============================================================
matrix_file = Path(f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_fan_out.txt")
json_dir    = Path("/home/gzu5140/Keerthana_b1042/grnInference/plot_data/f1_scores/")          # contains edges_1.json, edges_2.json, ...
# plot_dir    = Path("path/to/output/plots")
# plot_dir.mkdir(parents=True, exist_ok=True)

json_files = sorted(json_dir.glob("fan_out*.json"))


# ============================================================
# LOAD BINARY MATRIX (ONCE)
# ============================================================
gt = pd.read_csv(matrix_file, header=None)
n = gt.shape[0]
genes = [f"gene_{i+1}" for i in range(n)]
gt.index = genes
gt.columns = genes

gene_labels = [f"g{i+1}" for i in range(n)]

from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
import seaborn as sns
from pathlib import Path

def make_reds_blues_colormap(vmin=-0.05, vmax=0.18):
    """Custom red–white–blue colormap with pure white at 0, asymmetric."""
    # Calculate where 0 falls in the range [vmin, vmax]
    zero_position = (0 - vmin) / (vmax - vmin)
    
    # Number of colors for each segment (proportional to range)
    n_total = 256
    n_reds = int(zero_position * n_total)  # colors from vmin to 0
    n_blues = n_total - n_reds  # colors from 0 to vmax
    
    # Calculate intensity based on actual distance from zero
    # For reds: map from vmin to 0, so max intensity at vmin
    red_intensity = abs(vmin) / max(abs(vmin), abs(vmax))  # 0.05/0.18 ≈ 0.28
    # For blues: map from 0 to vmax, so max intensity at vmax  
    blue_intensity = abs(vmax) / max(abs(vmin), abs(vmax))  # 0.18/0.18 = 1.0
    
    # Create color arrays with scaled intensities
    reds = plt.cm.Reds(np.linspace(0.8 * red_intensity, 0, n_reds))  # scaled dark to light red
    whites = np.ones((1, 4))  # pure white at 0
    blues = plt.cm.Blues(np.linspace(0, 0.8 * blue_intensity, n_blues))  # light to scaled dark blue
    
    colors = np.vstack((reds, whites, blues))
    return LinearSegmentedColormap.from_list('RedsBlues', colors)


metrics_df_fan_out = []

# ============================================================
# MAIN LOOP
# ============================================================
for jf in json_files:

    # ---------------- load JSON ----------------
    with open(jf, "r") as f:
        cm = json.load(f)

    final_directed_edges = cm["final_directed_edges"]
    unfiltered_direction_matrix = pd.DataFrame(cm["unfiltered_direction_matrix"])
    multiple_states_and_reg = cm["gene_lists"]["multiple_states_and_reg"]
    all_gene_pairs = cm['all_gene_pairs']

    # enforce gene order
    unfiltered_direction_matrix.index = genes
    unfiltered_direction_matrix.columns = genes

    # ---------------- score ----------------
    # ============================================================
    # DIRECTED EDGE SCORING WITH ONE EXCEPTION (g2,g3)
    # ============================================================
    FP_edge = []
    FN_edge = []
    TP = FP = FN = TN = 0

    # quick lookup for predicted edges
    predicted_edges = set(tuple(e) for e in final_directed_edges)

    # special unordered exception pair
    EXCEPTION_PAIR = frozenset(["gene_2", "gene_3"])
    can_be_2_states =  [["gene_2", "gene_3"]]

    for i, gi in enumerate(genes):
        for j, gj in enumerate(genes):
            if gi == gj:
                continue

            pair = frozenset([gi, gj])

            # ----------------------------------------------------
            # EXCEPTION: (gene_2, gene_3)
            # ----------------------------------------------------
            if pair == EXCEPTION_PAIR:
                # evaluate ONCE per unordered pair
                if gi > gj:
                    continue  # avoid double counting

                has_gi_gj = (gi, gj) in predicted_edges
                has_gj_gi = (gj, gi) in predicted_edges

                if (has_gi_gj and has_gj_gi) or (not has_gi_gj and not has_gj_gi):
                    if [gi, gj] in multiple_states_and_reg or [gj, gi] in multiple_states_and_reg:
                        print(gi, gj, multiple_states_and_reg)
                        continue
                    else:
                        FP += 2
                        FP_edge.append((gi,gj))
                        FP_edge.append((gj, gi))
                else:
                    FP += 1
                    FP_edge.append((gi,gj))
                continue

            # ----------------------------------------------------
            # NORMAL DIRECTED SCORING
            # ----------------------------------------------------
            gt_edge = gt.loc[gi, gj] == 1
            pred_edge = (gi, gj) in predicted_edges

            if gt_edge and pred_edge:
                if [gi, gj] in multiple_states_and_reg and [gi, gj] not in can_be_2_states:
                    FN += 1
                    FN_edge.append((gi,gj))
                else:
                    TP += 1
            elif gt_edge and not pred_edge:
                FN += 1
                FN_edge.append((gi,gj))
            elif not gt_edge and pred_edge:
                FP += 1
                FP_edge.append((gi,gj))
            else:
                TN += 1

    # ============================================================
    # METRICS
    # ============================================================
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    print(f"TP={TP}, FP={FP}, FN={FN}, TN={TN}")
    print(f"Precision={precision:.4f}")
    print(f"Recall={recall:.4f}")
    print(f"F1={f1:.4f}")
    print(f"FP: {FP_edge}")
    print(f"FN: {FN_edge}")

    metrics_df_fan_out.append({
        "json": jf.name,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    #Plot TwINFER output
    gene_list = sorted(
        {g for pair in all_gene_pairs for g in pair},
        key=lambda x: int(x.split("_")[1])
    )
    gene_labels = [f"g{g.split('_')[1]}" for g in gene_list]
    # direction_matrix = unfiltered_direction_matrix.loc[gene_list, gene_list]
    data_matrix = unfiltered_direction_matrix.to_numpy(float)
    masked_matrix = np.fill_diagonal(data_matrix, np.nan)
    fig = plt.figure(figsize=(6,6))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.05, 0.95], hspace=0.1)
    cbar_ax = fig.add_subplot(gs[0])
    heatmap_ax = fig.add_subplot(gs[1])
    plot_matrix = data_matrix.copy()
    plot_matrix[:] = 0.0

    # Restore only final-ege correlations
    for g1, g2 in final_directed_edges:
        if g1 in gene_list and g2 in gene_list:
            i = gene_list.index(g1)
            j = gene_list.index(g2)
            plot_matrix[i, j] = data_matrix[i, j]
            plot_matrix[j, i] = data_matrix[j, i]
    np.fill_diagonal(plot_matrix, np.nan)
    # vmin = np.nanmin(plot_matrix)
    # vmax = np.nanmax(plot_matrix)
    vmin = 0.0
    vmax = 0.15
    cmap = make_reds_blues_colormap(vmin = vmin, vmax = vmax)

    cmap.set_bad(color="#D9D9D9")
    # --- draw heatmap ---
    sns.heatmap(
        plot_matrix,
        ax=heatmap_ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        square=True,
        cbar=True,
        cbar_ax=cbar_ax,
        cbar_kws={'orientation': 'horizontal'},
        linewidths=0.5,
        linecolor="black",
        # center=center
    )
    for g1, g2 in multiple_states_and_reg:
        if g1 in gene_list and g2 in gene_list:
            i = gene_list.index(g1)
            j = gene_list.index(g2)
            print(i, j)

            # Draw diagonal in cell (i, j) - top-left to bottom-right
            heatmap_ax.plot(
                [j, j+1],      # x: left → right
                [i, i+1],      # y: top → bottom
                linestyle="--",
                color="black",
                linewidth=1.5,
                clip_on=False
            )
            
            # Draw diagonal in symmetric cell (j, i) - top-left to bottom-right
            heatmap_ax.plot(
                [i, i+1],      # x: left → right
                [j, j+1],      # y: top → bottom
                linestyle="--",
                color="black",
                linewidth=1.5,
                clip_on=False
            )

    # --- labels ---
    cbar_ax.xaxis.set_label_position('top')
    cbar_ax.xaxis.tick_top()

    n = len(gene_labels)
    tick_pos = np.arange(n) + 0.5

    heatmap_ax.set_xticks(tick_pos)
    heatmap_ax.set_yticks(tick_pos)
    heatmap_ax.set_xticklabels(gene_labels, rotation=90, ha="center", va="top")
    heatmap_ax.set_yticklabels(gene_labels, rotation=0, ha="right", va="center")

    # keep limits tight to cells
    heatmap_ax.set_xlim(0, n)
    heatmap_ax.set_ylim(n, 0)

    # --- transparent background ---
    # plt.tight_layout()
    fig.patch.set_alpha(0)
    for ax in [heatmap_ax, cbar_ax]:
        ax.set_facecolor("none")
    for im in heatmap_ax.get_images() + cbar_ax.get_images():
        im.set_facecolor((1, 1, 1, 0))
        im.set_edgecolor((1, 1, 1, 0))
    for spine in heatmap_ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_edgecolor('black')
        spine.set_clip_on(False)

    plt.show()

# ============================================================
# METRICS SUMMARY
# ============================================================
metrics_df_fan_out = pd.DataFrame(metrics_df_fan_out)
print(metrics_df_fan_out)